In [1]:
import numpy as np
import warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.optimize import minimize
from scipy.stats import norm
warnings.filterwarnings('ignore')

In [2]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
print(data_in)
print(data_out)

X_init = data_in
y_init = data_out

[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837]


In [3]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because minimize()

# --- Optimize acquisition function with multiple random starts ---
y_best = y_init.max()
bounds = [(0,1), (0,1), (0,1)]

best_x = None
best_ei = float('inf')
for _ in range(10):
    x0 = np.random.rand(3)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next compound combination to try:", x_next)


Next compound combination to try: [0.31888668 0.29777486 0.18141117]


Week-02

In [4]:

X_init = np.array([
 [0.17152521, 0.34391687, 0.2487372 ],
 [0.24211446, 0.64407427, 0.27243281],
 [0.53490572, 0.39850092, 0.17338873],
 [0.49258141, 0.61159319, 0.34017639],
 [0.13462167, 0.21991724, 0.45820622],
 [0.34552327, 0.94135983, 0.26936348],
 [0.15183663, 0.43999062, 0.99088187],
 [0.64550284, 0.39714294, 0.91977134],
 [0.74691195, 0.28419631, 0.22629985],
 [0.17047699, 0.6970324 , 0.14916943],
 [0.22054934, 0.29782524, 0.34355534],
 [0.66601366, 0.67198515, 0.2462953 ],
 [0.04680895, 0.23136024, 0.77061759],
 [0.60009728, 0.72513573, 0.06608864],
 [0.96599485, 0.86111969, 0.56682913]]
)

y_init = np.array([-0.1121222 , -0.08796286, -0.11141465, -0.03483531, -0.04800758, -0.11062091,
 -0.39892551, -0.11386851, -0.13146061, -0.09418956, -0.04694741, -0.10596504,
 -0.11804826, -0.03637783, -0.05675837])

#week01's result
X_week2_add = np.array([1.000000, 1.000000, 0.837642])
y_week2_add = np.array([-0.07803969912604468])

X_all = np.vstack([X_init, X_week2_add])
y_all = np.concatenate([y_init, y_week2_add])

X_init = X_all
y_init = y_all

print(X_init)
print(y_init)

[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]
 [1.         1.         0.837642  ]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.0780397 ]


In [5]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

GaussianProcessRegressor(alpha=1e-06, kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)

In [6]:
# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because minimize()


In [7]:
# --- Optimize acquisition function with multiple random starts ---
y_best = y_init.max()
bounds = [(0,1), (0,1), (0,1)]

best_x = None
best_ei = float('inf')
for _ in range(30):
    x0 = np.random.rand(3)
    res = minimize(lambda x: expected_improvement(x, gp, y_best, xi=0.01),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next compound combination to try:", x_next)

Next compound combination to try: [0.82888664 0.34793238 0.5545142 ]


Week-03

In [9]:
X_init = np.array([
 [0.17152521, 0.34391687, 0.2487372 ],
 [0.24211446, 0.64407427, 0.27243281],
 [0.53490572, 0.39850092, 0.17338873],
 [0.49258141, 0.61159319, 0.34017639],
 [0.13462167, 0.21991724, 0.45820622],
 [0.34552327, 0.94135983, 0.26936348],
 [0.15183663, 0.43999062, 0.99088187],
 [0.64550284, 0.39714294, 0.91977134],
 [0.74691195, 0.28419631, 0.22629985],
 [0.17047699, 0.6970324 , 0.14916943],
 [0.22054934, 0.29782524, 0.34355534],
 [0.66601366, 0.67198515, 0.2462953 ],
 [0.04680895, 0.23136024, 0.77061759],
 [0.60009728, 0.72513573, 0.06608864],
 [0.96599485, 0.86111969, 0.56682913]]
)

y_init = np.array([-0.1121222 , -0.08796286, -0.11141465, -0.03483531, -0.04800758, -0.11062091,
 -0.39892551, -0.11386851, -0.13146061, -0.09418956, -0.04694741, -0.10596504,
 -0.11804826, -0.03637783, -0.05675837])

#week01's result
X_init = np.vstack([X_init, [1.000000, 1.000000, 0.837642]])
y_init = np.concatenate([y_init, [-0.07803969912604468]])

#week02's result
X_init = np.vstack([X_init, [0.429230,0.961997,0.104852]])
y_init = np.concatenate([y_init, [-0.08417946188382879]])

print(X_init)
print(y_init)


[[0.17152521 0.34391687 0.2487372 ]
 [0.24211446 0.64407427 0.27243281]
 [0.53490572 0.39850092 0.17338873]
 [0.49258141 0.61159319 0.34017639]
 [0.13462167 0.21991724 0.45820622]
 [0.34552327 0.94135983 0.26936348]
 [0.15183663 0.43999062 0.99088187]
 [0.64550284 0.39714294 0.91977134]
 [0.74691195 0.28419631 0.22629985]
 [0.17047699 0.6970324  0.14916943]
 [0.22054934 0.29782524 0.34355534]
 [0.66601366 0.67198515 0.2462953 ]
 [0.04680895 0.23136024 0.77061759]
 [0.60009728 0.72513573 0.06608864]
 [0.96599485 0.86111969 0.56682913]
 [1.         1.         0.837642  ]
 [0.42923    0.961997   0.104852  ]]
[-0.1121222  -0.08796286 -0.11141465 -0.03483531 -0.04800758 -0.11062091
 -0.39892551 -0.11386851 -0.13146061 -0.09418956 -0.04694741 -0.10596504
 -0.11804826 -0.03637783 -0.05675837 -0.0780397  -0.08417946]


In [13]:
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu = mu[0]
    sigma = sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative because minimize()

# --- Optimize acquisition function ---
bounds = [(0,1), (0,1), (0,1)]
y_best = y_init.max()

best_x = None
best_ei = float('inf')
for _ in range(50):
    x0 = np.random.rand(3)
    res = minimize(lambda x: expected_improvement(x, gp, y_best, xi=0.01),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)


0.491221-0.812746-0.787239
